In [153]:
import pandas as pd
import re

In [154]:
df = pd.read_csv("chotot_raw.csv")

df.head()

,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,...,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"2,38 tỷ- 100 m2",- 100 m2,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đườ...,100 m2,"23,8 triệu/m2",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,18 tỷ- 79 m2,- 79 m2,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","Nhà 1 trệt 2 lầu\nDiện tích 4,15x18,8\n4 phòng...",79 m²,"227,85 triệu/m²",Nam,Đang chờ sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1 tỷ- 500 m2,- 500 m2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \n...",500 m2,2 triệu/m2,Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn",525 triệu- 60 m2,- 60 m2,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...","Nhà chính chủ mới xây đường võ văn vân, vĩnh l...",60 m²,"8,75 triệu/m²",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,440 triệu- 150 m2,- 150 m2,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...","bán lô đất đẹp mặt tiền đường nhựa thôn 2, Suố...",150 m2,"2,93 triệu/m2",Bắc,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Location Column

## Hướng xử lý
### 📍 Location Parsing Rules

#### 🎯 Mục tiêu
Chuẩn hóa và tách trường `location` thành 4 thành phần:

- `street` (đường)
- `ward` (phường/xã)
- `district` (quận/huyện)
- `city` (tỉnh/thành phố)

---

### 🧩 1. Tiền xử lý (Preprocessing)

- Chuẩn hóa text:
  - chuyển về chữ thường
  - loại bỏ khoảng trắng dư
- Tách chuỗi theo dấu phẩy `,`
- Loại bỏ phần tử đầu nếu là số (ví dụ: `"36"`)

---

### 🧠 2. Nhận diện theo rule (Rule-based classification)

Mỗi phần tử sau khi split sẽ được phân loại dựa trên **từ đứng đầu (prefix)**.

---

#### 🟢 Street (Đường)

Nếu phần tử bắt đầu bằng:

- `đường`

→ Gán vào `street`

---

#### 🟡 Ward (Phường/Xã)

Nếu phần tử bắt đầu bằng:

- `phường`
- `xã`
- `thị trấn`
- `thôn`
- `kênh`

→ Gán vào `ward`

---

#### 🔵 District (Quận/Huyện)

Nếu phần tử bắt đầu bằng:

- `quận`
- `huyện`
- `thị xã`

→ Gán vào `district`

---

#### 🔴 City (Tỉnh/Thành phố)

Nếu phần tử bắt đầu bằng:

- `tỉnh`
- `tp`

→ Gán vào `city`

---

#### ⚠️ Trường hợp đặc biệt: `thành phố`

- Nếu chưa có `district` → gán vào `district`
- Nếu đã có `district` → gán vào `city`

---

### 🔄 3. Fallback Rule

Nếu chưa xác định được `city`:

- Nếu số phần tử ≥ 3  
→ lấy phần tử cuối cùng làm `city`

---

### 🔁 4. Fill các field còn thiếu

- Loại bỏ các phần tử đã được sử dụng
- Các phần còn lại sẽ được gán theo thứ tự:


In [155]:
df[["location"]].head(10)

,location
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ..."
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu..."
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch..."
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ..."
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ..."
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức,..."
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả..."
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả..."
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch..."


In [156]:
def clean_location(text):
    if pd.isna(text):
        return None
    
    text = text.lower().strip()
    
    # bỏ ký tự rác
    text = re.sub(r"\|\|\d+", "", text)
    
    # normalize space
    text = re.sub(r"\s+", " ", text)
    
    return text

df["location_clean"] = df["location"].apply(clean_location)

df[["location", "location_clean"]].head(10)

,location,location_clean
0,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...","đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ..."
1,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","đường cao đạt, phường 1, quận 5, tp hồ chí minh"
2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...","đường quốc lộ 13, thị trấn lai uyên, huyện bàu..."
3,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...","đường võ văn vân, xã vĩnh lộc b, huyện bình ch..."
4,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...","thôn 2, xã suối rao, huyện châu đức, bà rịa - ..."
5,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...","đường lý thái tổ, xã đạm bri, thành phố bảo lộ..."
6,"Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...","đường số 1, phường trường thọ, quận thủ đức,..."
7,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...","đường quốc lộ 20, thị trấn lộc thắng, huyện bả..."
8,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...","đường 359, xã tân dương, huyện thuỷ nguyên, hả..."
9,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...","đường phó cơ điều, phường 12, quận 5, tp hồ ch..."


In [157]:
def debug_split(text):
    if pd.isna(text):
        return None
    return [p.strip() for p in text.split(",")]

df["parts"] = df["location_clean"].apply(debug_split)

df[["location_clean", "parts"]].head(10)

,location_clean,parts
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ...","[đường lỗ giáng 8, phường hòa xuân, quận cẩm l..."
1,"đường cao đạt, phường 1, quận 5, tp hồ chí minh","[đường cao đạt, phường 1, quận 5, tp hồ chí minh]"
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu...","[đường quốc lộ 13, thị trấn lai uyên, huyện bà..."
3,"đường võ văn vân, xã vĩnh lộc b, huyện bình ch...","[đường võ văn vân, xã vĩnh lộc b, huyện bình c..."
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - ...","[thôn 2, xã suối rao, huyện châu đức, bà rịa -..."
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộ...","[đường lý thái tổ, xã đạm bri, thành phố bảo l..."
6,"đường số 1, phường trường thọ, quận thủ đức,...","[đường số 1, phường trường thọ, quận thủ đức..."
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bả...","[đường quốc lộ 20, thị trấn lộc thắng, huyện b..."
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hả...","[đường 359, xã tân dương, huyện thuỷ nguyên, h..."
9,"đường phó cơ điều, phường 12, quận 5, tp hồ ch...","[đường phó cơ điều, phường 12, quận 5, tp hồ c..."


In [158]:
import unicodedata
import re

def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [159]:
def starts_with_any(text, keywords):
    return any(text.startswith(k + " ") or text == k for k in keywords)


def parse_location(text):
    if pd.isna(text):
        return pd.Series({
            "street": None,
            "ward": None,
            "district": None,
            "city": None
        })
    
    # 🔥 normalize
    text = normalize_text(text)
    
    # 🔹 split
    parts = [p.strip() for p in text.split(",") if p.strip() != ""]
    
    # 🔹 remove số đầu (vd: "36")
    if len(parts) > 1 and re.fullmatch(r"\d+", parts[0]):
        parts = parts[1:]
        # print("PARTS:", parts)  # debug nếu cần
    
    result = {
        "street": None,
        "ward": None,
        "district": None,
        "city": None
    }
    
    unknown_parts = []
    
    # 🔥 1. Keyword detection (prefix-based)
    for part in parts:
        p = part.strip()
        
        if starts_with_any(p, ["phường", "xã", "thị trấn", "thôn", "kênh"]):
            result["ward"] = part
        
        elif starts_with_any(p, ["quận", "huyện", "thị xã"]):
            result["district"] = part
        
        elif starts_with_any(p, ["đường"]):
            result["street"] = part
        
        elif starts_with_any(p, ["tỉnh", "tp"]):
            result["city"] = part
        
        elif p.startswith("thành phố"):
            if result["district"] is None:
                result["district"] = part
            else:
                result["city"] = part
        
        else:
            unknown_parts.append(part)   # ✅ FIX INDENT
    
    # 🔥 2. Fix city (fallback)
    if result["city"] is None and len(parts) >= 3:
        result["city"] = parts[-1]
    
    # 🔥 3. Remove các phần đã dùng
    used = set([v for v in result.values() if v is not None])
    remaining = [p for p in parts if p not in used]
    
    # 🔥 4. Fill phần còn lại
    for key in ["street", "ward", "district"]:
        if result[key] is None and remaining:
            result[key] = remaining.pop(0)
    
    return pd.Series(result)

In [160]:
df[["street", "ward", "district", "city"]] = df["location_clean"].apply(parse_location)

df[["location_clean", "street", "ward", "district", "city"]].head(30)

,location_clean,street,ward,district,city
0,"đường lỗ giáng 8, phường hòa xuân, quận cẩm lệ...",đường lỗ giáng 8,phường hòa xuân,quận cẩm lệ,đà nẵng
1,"đường cao đạt, phường 1, quận 5, tp hồ chí minh",đường cao đạt,phường 1,quận 5,tp hồ chí minh
2,"đường quốc lộ 13, thị trấn lai uyên, huyện bàu...",đường quốc lộ 13,thị trấn lai uyên,huyện bàu bàng,bình dương
3,"đường võ văn vân, xã vĩnh lộc b, huyện bình ch...",đường võ văn vân,xã vĩnh lộc b,huyện bình chánh,tp hồ chí minh
4,"thôn 2, xã suối rao, huyện châu đức, bà rịa - ...",thôn 2,xã suối rao,huyện châu đức,bà rịa - vũng tàu
5,"đường lý thái tổ, xã đạm bri, thành phố bảo lộ...",đường lý thái tổ,xã đạm bri,thành phố bảo lộc,lâm đồng
6,"đường số 1, phường trường thọ, quận thủ đức,...",đường số 1,phường trường thọ,quận thủ đức,tp hồ chí minh
7,"đường quốc lộ 20, thị trấn lộc thắng, huyện bả...",đường quốc lộ 20,thị trấn lộc thắng,huyện bảo lâm,lâm đồng
8,"đường 359, xã tân dương, huyện thuỷ nguyên, hả...",đường 359,xã tân dương,huyện thuỷ nguyên,hải phòng
9,"đường phó cơ điều, phường 12, quận 5, tp hồ ch...",đường phó cơ điều,phường 12,quận 5,tp hồ chí minh


In [161]:
test = "Thôn Lộc Châu 2, Xã Tân Nghĩa, Huyện Di Linh, Lâm Đồng"

print(parse_location(clean_location(test)))

street      thôn lộc châu 2
ward           xã tân nghĩa
district      huyện di linh
city               lâm đồng
dtype: str


In [162]:
df[df["city"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [163]:
df[df["district"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [164]:
df[df["ward"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
31,Đường Nguyễn Văn Bứa,đường nguyễn văn bứa,NaN,NaN,NaN
50,Đường Phạm Văn Đồng,đường phạm văn đồng,NaN,NaN,NaN
116,Đường Đinh Đức Thiện,đường đinh đức thiện,NaN,NaN,NaN
165,Đường D7,đường d7,NaN,NaN,NaN
210,Đường Mạc Đăng Doanh,đường mạc đăng doanh,NaN,NaN,NaN
261,Đường Bùi Đình Túy,đường bùi đình túy,NaN,NaN,NaN
368,Hương An,hương an,NaN,NaN,NaN
414,D6,d6,NaN,NaN,NaN
679,Đường D7,đường d7,NaN,NaN,NaN
785,Duong go xoai,duong go xoai,NaN,NaN,NaN


In [165]:
df[df["street"].isna()][["location", "street", "ward", "district", "city"]].head(20)

,location,street,ward,district,city
834,"200, Xã Bình Hiệp, Huyện Bình Sơn, Quảng Ngãi",NaN,xã bình hiệp,huyện bình sơn,quảng ngãi
1394,"301, Xã Tân Thạnh Đông, Huyện Củ Chi, Tp Hồ Ch...",NaN,xã tân thạnh đông,huyện củ chi,tp hồ chí minh
1693,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",NaN,xã tân phước,huyện đồng phú,bình phước
1719,"322, Xã Tân Phước, Huyện Đồng Phú, Bình Phước",NaN,xã tân phước,huyện đồng phú,bình phước
2120,"xã Duy Nghĩa, Xã Duy Nghĩa, Huyện Duy Xuyên, Q...",NaN,xã duy nghĩa,huyện duy xuyên,quảng nam
2327,"980, Phường Phú Hữu, Quận 9, Tp Hồ Chí Minh",NaN,phường phú hữu,quận 9,tp hồ chí minh
2526,"1, Xã An Linh, Huyện Phú Giáo, Bình Dương",NaN,xã an linh,huyện phú giáo,bình dương
2532,"943, Xã Vĩnh Thành, Huyện Châu Thành, An Giang",NaN,xã vĩnh thành,huyện châu thành,an giang
2791,"713, Thị trấn Đạ M'ri, Huyện Đạ Huoai, Lâm Đồng",NaN,thị trấn đạ m'ri,huyện đạ huoai,lâm đồng
3331,"105, Phường Tân Phú, Quận 9, Tp Hồ Chí Minh",NaN,phường tân phú,quận 9,tp hồ chí minh


In [166]:
parse_location("Đường Tỉnh lộ 15")

street      đường tỉnh lộ 15
ward                     NaN
district                 NaN
city                     NaN
dtype: str

In [167]:
df.columns

Index(['title', 'price', 'area', 'location', 'description', 'Diện tích đất:',
       'Giá/m2:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:',
       'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:',
       'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:',
       'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:',
       'Loại hình căn hộ:', 'Tổng số tầng:', 'Tên phân khu/Lô/Block/Tháp:',
       'Mã căn / Mã căn hộ:', 'Tầng số:', 'Hướng ban công:', 'Mã lô:',
       'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'location_clean', 'parts',
       'street', 'ward', 'district', 'city'],
      dtype='str')

# Check value, unique, ununique, missing data của các column category

In [168]:
def check_category(df, col_name, top_n=10):
    print(f"=== COLUMN: {col_name} ===")
    
    # 1. Số category
    print("\n🔢 Number of unique values:")
    print(df[col_name].nunique())
    
    # 2. Số missing
    print("\n⚠️ Missing values:")
    print(df[col_name].isna().sum())
    
    # 3. Top value phổ biến
    print(f"\n📊 Top {top_n} values:")
    print(df[col_name].value_counts().head(top_n))
    
    # 4. List unique (optional)
    print("\n📋 Sample unique values:")
    print(df[col_name].dropna().unique()[:top_n])

In [169]:
cols = ["Hướng cửa chính:",'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Tình trạng bất động sản:','Loại hình căn hộ:', 'Tên phân khu/Lô/Block/Tháp:','Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:']

for col in cols:
    check_category(df, col)

=== COLUMN: Hướng cửa chính: ===

🔢 Number of unique values:
8

⚠️ Missing values:
0

📊 Top 10 values:
Hướng cửa chính:
Đông Nam    2123
Tây Nam     1181
Tây Bắc     1160
Đông Bắc    1034
Đông        1019
Nam          932
Bắc          825
Tây          730
Name: count, dtype: int64

📋 Sample unique values:
<StringArray>
['Nam', 'Bắc', 'Đông Bắc', 'Tây Nam', 'Đông Nam', 'Đông', 'Tây', 'Tây Bắc']
Length: 8, dtype: str
=== COLUMN: Giấy tờ pháp lý: ===

🔢 Number of unique values:
3

⚠️ Missing values:
0

📊 Top 10 values:
Giấy tờ pháp lý:
Đã có sổ        8325
Đang chờ sổ      437
Giấy tờ khác     242
Name: count, dtype: int64

📋 Sample unique values:
<StringArray>
['Đã có sổ', 'Đang chờ sổ', 'Giấy tờ khác']
Length: 3, dtype: str
=== COLUMN: Đặc điểm nhà/đất: ===

🔢 Number of unique values:
3

⚠️ Missing values:
0

📊 Top 10 values:
Đặc điểm nhà/đất:
Hẻm xe hơi    3850
Nở hậu        3009
Mặt tiền      2145
Name: count, dtype: int64

📋 Sample unique values:
<StringArray>
['Mặt tiền', 'Nở hậu', 

- loại hình nhà ở: 1 missing value
- Tình trạng nội thất: 1 missing value
- Tình hình bất động sản: 6 missing value
- Tình hình loại căn hộ: 6 missing value
- Tên phân khu/lô/block/tháp:35 missing value
- Hướng ban công: 47 missing value
- Đặc điểm căn hộ:69 missing value
- Loại văn phòng: 291 missing value

Vì các giá trị trên liên quan đến đặc điểm nhà đất, nếu giá trị NA phù hợp thì có thể pass

In [170]:
cols = [
    "Đặc điểm nhà/đất:",
    "Loại hình căn hộ:",
    "Tổng số tầng:",
    "Tên phân khu/Lô/Block/Tháp:",
    "Mã căn / Mã căn hộ:",
    "Tầng số:",
    "Hướng ban công:",
    "Mã lô:",
    "Đặc điểm căn hộ:",
    "Loại hình văn phòng:"
]

In [171]:
def check_na_with_related_cols(df, target_col, related_cols, n=20):
    # điều kiện NaN hoặc empty string
    condition = (
        df[target_col].isna() |
        (df[target_col].astype(str).str.strip() == "")
    )
    
    # chọn cột cần xem
    cols_to_show = ["title", "location", target_col] + related_cols
    
    result = df.loc[condition, cols_to_show]
    
    print(f"🔎 Total missing rows in '{target_col}': {len(result)}")
    
    return result.head(n)

# Loại hình nhà ở Column

In [172]:
check_na_with_related_cols(df, "Loại hình nhà ở:", cols)

🔎 Total missing rows in 'Loại hình nhà ở:': 1


,title,location,Loại hình nhà ở:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Đối với data trên, có thể khả năng rơi vào tình trạng là đất chưa xây

# Tình trạng nội thất Column

In [173]:
check_na_with_related_cols(df, "Tình trạng nội thất:", cols)

🔎 Total missing rows in 'Tình trạng nội thất:': 1


,title,location,Tình trạng nội thất:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Tình hình nội thất cũng NA, dữ liệu này dữ lại để phân tích đối với trường hợp đất chưa xây nhà.

# Tình trạng bất động sản Column

In [174]:
check_na_with_related_cols(df, "Tình trạng bất động sản:", cols)

🔎 Total missing rows in 'Tình trạng bất động sản:': 6


,title,location,Tình trạng bất động sản:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Loại hình căn hộ Column

In [175]:
check_na_with_related_cols(df, "Loại hình căn hộ:", cols)

🔎 Total missing rows in 'Loại hình căn hộ:': 6


,title,location,Loại hình căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Đánh giá dữ liệu: 
+ No0: Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu - đã đánh giá --> Đất chưa xây nhà 
+ No1: Nhà mặt tiền 1 trệt 2 lầu - dữ liệu các cột Tổng số tầng, mã căn/mã căn hộ không có giá trị --> missing data --> loại bỏ
+ No2: Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2	- dữ liệu OK, rơi vào tình trạng đất chưa xây
+ No3: Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn - dữ liệu title Nhà đẹp nhưng không có giá trị khách --> missing data --> Loại bỏ
+ No4: Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao	- dữ liệu OK, rơi vào tình trạng đất chưa xây
+ No5: 600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...	- dữ liệu OK, rơi vào tình trạng đất chưa xây

Từ thông tin đánh gái trên, loại bỏ các giá trị từ No1, No3

In [217]:
df = df[
    ~(
        df["Loại hình căn hộ:"].isna() & 
        df["title"].str.contains("Nhà", case=False, na=False)
    )
]

In [218]:
check_na_with_related_cols(df, "Loại hình căn hộ:", cols)

🔎 Total missing rows in 'Loại hình căn hộ:': 4


,title,location,Loại hình căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,other,NaN,other,other


# Tên phân khu/Lô/Block/Tháp Column

## 📍 Feature Engineering: Tên phân khu / Lô / Block / Tháp

### 🎯 Mục tiêu

Chuẩn hóa và phân loại dữ liệu từ cột: Tên phân khu/Lô/Block/Tháp:

thành 3 nhóm chính:

- `project` → tên dự án / khu dân cư
- `block` → ký hiệu block/lô (A, B, A1, Lô E…)
- `other` → dữ liệu không hợp lệ / mô tả / nhiễu

---

### 🧩 1. Đặc điểm dữ liệu

Cột này là **mixed data**, bao gồm nhiều loại thông tin khác nhau:

#### 🟢 Project (giá trị hợp lệ)
- KHU DÂN CƯ AN PHÚ HƯNG
- SÀI GÒN VILLAGE
- MANHATTAN
- SONADEZI CHÂU ĐỨC

---

#### 🟡 Block / Lô (ký hiệu)A
- B
- A1
- B2
- LÔ E
- BLOCK B
- A B
- A B C

---

#### 🔴 Other (nhiễu / sai ngữ cảnh)
- NHÀ ĐẸP GẦN CÔNG VIÊN
- ĐẤT CHÍNH CHỦ
- PHẠM VĂN ĐỒNG
- 12345

---

### 🧠 2. Pipeline xử lý
- Clean → Classify → Normalize → Assign

---

### 🔹 3. Bước 1: Chuẩn hóa dữ liệu (Cleaning)

### Rule:

- Chuyển toàn bộ về chữ hoa
- Loại bỏ khoảng trắng dư

### Ví dụ:

| Input | Output |
|------|--------|
| `a b` | `A B` |
| `  block b  ` | `BLOCK B` |

---

### 🔹 4. Bước 2: Phân loại (Classification)

#### 🟢 Project

Nếu chuỗi chứa một trong các keyword:KHU, DỰ ÁN, VILLAGE, CITY, RESIDENCE, RESIDENCES, PARADISE, RIVERSIDE, URBAN, KDC

→ Gán: `project`

---

### 🟡 Block

Nếu match một trong các pattern sau:

#### Pattern 1: ký hiệu đơn
A, B, C, A1, B12

Regex:[A-Z]\d{0,2}

---

#### Pattern 2: có prefix
LÔ A
BLOCK B

Regex: (LÔ|BLOCK)\s*[A-Z0-9]+

---

#### Pattern 3: nhiều block
A B
A B C

Regex:A-Z--> +

→ Gán: `block`

---

### 🔴 Other

Các trường hợp còn lại:

- mô tả tự do
- địa chỉ
- dữ liệu số
- dữ liệu nhiễu

→ Gán: `other`

---

## 🔹 5. Bước 3: Chuẩn hóa Block (Normalization)

### Rule:

- Tách theo khoảng trắng
- Loại bỏ giá trị trùng
- Sắp xếp lại
- Join bằng dấu phẩy `,`

### Ví dụ:

| Input | Output |
|------|--------|
| `A B` | `A,B` |
| `B A` | `A,B` |
| `A A B` | `A,B` |

---

## 🔹 6. Bước 4: Tạo cột cuối (`ten_phan_khu_final`)

| Loại | Xử lý |
|------|------|
| project | giữ nguyên |
| block | normalize thành format chuẩn |
| other | `"Không thuộc project/block"` |

---

## 🧪 7. Ví dụ

### Input
A B

### Output
phan_loai = block
ten_phan_khu_final = A,B
---

### Input
KHU DÂN CƯ AN PHÚ HƯNG

### Output
phan_loai = project
ten_phan_khu_final = KHU DÂN CƯ AN PHÚ HƯNG


---

## ⚠️ 8. Lưu ý quan trọng

### ❗ Đây không phải categorical thuần

Cột này chứa:

- entity (tên dự án)
- code (block)
- text noise

→ cần xử lý theo rule-based pipeline

---

### ❗ Rule-based hiệu quả hơn ML

Do dữ liệu nhiễu cao → dùng rule-based giúp kiểm soát tốt hơn

---

### ❗ Có thể mở rộng

- thêm keyword project
- refine regex block
- chuẩn hóa tên dự án

---

## 🚀 9. Kết luận

Pipeline xử lý:Clean → Classify → Normalize → Assign

Giúp:

- giảm nhiễu dữ liệu
- chuẩn hóa category
- tăng chất lượng feature cho mô hình ML



In [176]:
check_na_with_related_cols(df, "Tên phân khu/Lô/Block/Tháp:", cols)

🔎 Total missing rows in 'Tên phân khu/Lô/Block/Tháp:': 35


,title,location,Tên phân khu/Lô/Block/Tháp:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [219]:
# Clear Data
def clean_text(x):
    if pd.isna(x):
        return x
    x = x.upper().strip()
    x = re.sub(r"\s+", " ", x)
    return x

df["ten_phan_khu_clean"] = df["Tên phân khu/Lô/Block/Tháp:"].apply(clean_text)


In [220]:
df["ten_phan_khu_clean"].value_counts()

ten_phan_khu_clean
A                                1066
B                                 728
THÔN BÌNH KHÁNH                   275
1                                 239
C                                 222
                                 ... 
L                                   1
THỬA 820                            1
CAM LÂM, NHA TRANG, KHÁNH HÒA       1
103 NGUYỄN THỊ THÂPH                1
LÔ B                                1
Name: count, Length: 244, dtype: int64

In [221]:
for val in df["ten_phan_khu_clean"].dropna().unique():
    print(val)

DỰ ÁN TÍN HƯNG
B
KHU DÂN CƯ AN PHÚ HƯNG
BỆNH VIỆN XUYÊN Á TÂY NINH
A
C
PHƯỚC BÌNH
ABCD
AN BÌNH
BAU BÀNG
KHU ĐƯỜNG BÀN CỜ PHÚ THẠNH-PHÚ THỌ HÒA
L43, L44, L45
RIGEL
D1
115
ĐẤT TÂN HIỆP
KBD
THÔN BÌNH KHÁNH
KDC
DỊCH VỤ 6.9
D
SÀI GÒN VILLAGE
BABYLON
NHÀ DOI DIEN CONG VIEN,RAT MAT ME.
MP1
B3
KHU ĐÔ THỊ TRƯỜNG AN
A,B
DCH
NHÀ PHỐ LIỀN KỀ
E3
KHU HOÀNG HOA THÁM
A B
KHU DÂN CU
1
HOMELAND PARADISE VILLA
2
TOM 77
H
KDCD
NAM CẨM LỆ
MANHATTAN
C4,C5,A4
N16
H20
HẺM 481 ĐƯỜNG TÂN KỲ TÂN QUÝ
HANEL SÀI ĐỒNG LONG BIÊN
03
749
E
NGỌC ĐỊNH FARM
T5
LÔ E
LIỀN KHU DÂN CƯ THỊNH VƯỢNG
CHÂU THỚI
ĐẠI THÀNH NGHI KIM
NGÕ 37 ĐẠI ĐỒNG
DỰ ÁN KHU NGHĨ DƯỠNG BÃI DÀI PHÚ QUỐC
00
A8
BLOCK B
VĨNH ĐIỀM THƯỢNG
ẤP 1 SÔNG TRẦU THỊ TRẤN TRẢNG BOM
KHU DÂN CƯ
KHU DÂN CƯ THẠNH MỸ LỢI DRAGON
BÌNH KHÁNH 2
S2.02
ĐẢO THỊNH VƯỢNG TAM ĐA
ĐƯỜNG 4A
TA15
THE MANHATTAN GLORY
BÌNH NGUYÊN
A1
KHU DAN CƯ CAO CÂP
6-MAY
MẶT TIỀN 833
G50
HẺM TRỊNH ĐÌNH TRỌNG
L K J H F G
CHÙA ĐỨC VIÊN
B2.11
965
NHÀ HẺM
NAM LONG
01
KHU DÂN CƯ NINH GIANG CÁT LÁI
CT1
MẶT

In [223]:
# Classify Data
def classify_type(x):
    if pd.isna(x):
        return "other"
    
    if any(k in x for k in [
        "KHU", "DỰ ÁN", "VILLAGE", "CITY", "RESIDENCE", "RESIDENCES",
        "PARADISE", "RIVERSIDE", "URBAN", "KDC"
    ]):
        return "project"
    
    if re.fullmatch(r"[A-Z]\d{0,2}", x):
        return "block"
    
    if re.fullmatch(r"(LÔ|BLOCK)\s*[A-Z0-9]+", x):
        return "block"
    
    if re.fullmatch(r"[A-Z](\s+[A-Z0-9]+)+", x):
        return "block"
    
    return "other"

In [224]:
df["phan_loai"] = df["ten_phan_khu_clean"].apply(classify_type)

In [226]:
# NORMALIZE BLOCK
def normalize_block(x):
    if pd.isna(x):
        return x
    parts = re.split(r"\s+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

In [200]:
def normalize_block(x):
    parts = re.split(r"[,\s]+", x)
    parts = sorted(set(parts))
    return ",".join(parts)

df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

In [228]:
# FINAL COLUMN
df["ten_phan_khu_final"] = None

# project giữ nguyên
df.loc[df["phan_loai"] == "project", "ten_phan_khu_final"] = df["ten_phan_khu_clean"]

# block normalize
df.loc[df["phan_loai"] == "block", "ten_phan_khu_final"] = (
    df.loc[df["phan_loai"] == "block", "ten_phan_khu_clean"]
    .apply(normalize_block)
)

# other
df.loc[df["phan_loai"] == "other", "ten_phan_khu_final"] = "Không thuộc project/block"

In [229]:
df[["Tên phân khu/Lô/Block/Tháp:","ten_phan_khu_clean", "phan_loai", "ten_phan_khu_final"]].head(40)

,Tên phân khu/Lô/Block/Tháp:,ten_phan_khu_clean,phan_loai,ten_phan_khu_final
0,NaN,NaN,other,Không thuộc project/block
2,NaN,NaN,other,Không thuộc project/block
4,NaN,NaN,other,Không thuộc project/block
5,NaN,NaN,other,Không thuộc project/block
6,NaN,NaN,other,Không thuộc project/block
7,NaN,NaN,other,Không thuộc project/block
8,NaN,NaN,other,Không thuộc project/block
9,NaN,NaN,other,Không thuộc project/block
10,NaN,NaN,other,Không thuộc project/block
11,NaN,NaN,other,Không thuộc project/block


# Hướng ban công Column

In [203]:
check_na_with_related_cols(df, "Hướng ban công:", cols)

🔎 Total missing rows in 'Hướng ban công:': 47


,title,location,Hướng ban công:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [204]:
df["Hướng ban công:"] = (
    df["Hướng ban công:"]
    .fillna("other")
    .astype(str)
    .str.strip()
    .replace("", "other")
)

In [205]:
df["Hướng ban công:"].value_counts()

Hướng ban công:
Tây Bắc     2388
Đông Nam    2303
Đông Bắc    1686
Bắc          655
Nam          629
Tây Nam      498
Đông         482
Tây          316
other         47
Name: count, dtype: int64

Other ở hướng ban công là giá trị NaN --> missing data hoặc là không có ban công (nhà đã xây), nhà chưa xây

# Đặc điểm căn hộ Column

In [206]:
check_na_with_related_cols(df, "Đặc điểm căn hộ:", cols)

🔎 Total missing rows in 'Đặc điểm căn hộ:': 69


,title,location,Đặc điểm căn hộ:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,other,NaN,NaN,NaN


In [207]:
df["Đặc điểm căn hộ:"] = (
    df["Đặc điểm căn hộ:"]
    .fillna("other")
    .astype(str)
    .str.strip()
    .replace("", "other")
)

In [208]:
df["Đặc điểm căn hộ:"].value_counts()

Đặc điểm căn hộ:
Căn góc    8935
other        69
Name: count, dtype: int64

Other ở Đặc điểm căn hộ là giá trị NaN --> missing data hoặc là không có ban công (nhà đã xây), nhà chưa xây

# Loại hình văn phòng Column

In [209]:
check_na_with_related_cols(df, "Loại hình văn phòng:", cols)

🔎 Total missing rows in 'Loại hình văn phòng:': 291


,title,location,Loại hình văn phòng:,Đặc điểm nhà/đất:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2","Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn","Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...",NaN,Mặt tiền,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
5,600m2 đất thổ cư SHR cách đường Lý Thái Tổ Bảo...,"Đường Lý Thái Tổ, Xã Đạm Bri, Thành phố Bảo Lộ...",NaN,Nở hậu,NaN,NaN,NaN,NaN,NaN,other,NaN,other,NaN
6,"Kẹt vốn bán căn 2PN view hồ bơi tầng 10 giá 2,...","Đường số 1, Phường Trường Thọ, Quận Thủ Đức,...",NaN,Nở hậu,Chung cư,NaN,NaN,NaN,NaN,other,NaN,other,NaN
7,Bán đất mặt tiền 1 sẹt Quốc Lộ 20,"Đường Quốc Lộ 20, Thị trấn Lộc Thắng, Huyện Bả...",NaN,Mặt tiền,Chung cư,NaN,NaN,NaN,NaN,other,NaN,other,NaN
8,Chính chủ cần bán đất xã Liên Khê,"Đường 359, Xã Tân Dương, Huyện Thuỷ Nguyên, Hả...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,other,NaN,other,NaN
9,Nhà trệt 4 lầu sân thượng (4x20)p12.q5.25tỷ TL,"Đường Phó Cơ Điều, Phường 12, Quận 5, Tp Hồ Ch...",NaN,Hẻm xe hơi,Chung cư,NaN,NaN,NaN,NaN,other,NaN,other,NaN


In [210]:
df["Loại hình văn phòng:"] = (
    df["Loại hình văn phòng:"]
    .fillna("other")
    .astype(str)
    .str.strip()
    .replace("", "other")
)

In [211]:
df["Loại hình văn phòng:"].value_counts()

Loại hình văn phòng:
Mặt bằng kinh doanh    6078
Shophouse              2454
other                   291
Văn phòng               181
Name: count, dtype: int64

Other ở Loại hình văn phòng là giá trị NaN --> missing data hoặc là không có ban công (nhà đã xây), nhà chưa xây

In [231]:
df = pd.read_csv("chotot_raw.csv")

df.head()

,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,...,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"2,38 tỷ- 100 m2",- 100 m2,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ...",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đườ...,100 m2,"23,8 triệu/m2",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm qu...,18 tỷ- 79 m2,- 79 m2,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","Nhà 1 trệt 2 lầu\nDiện tích 4,15x18,8\n4 phòng...",79 m²,"227,85 triệu/m²",Nam,Đang chờ sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1 tỷ- 500 m2,- 500 m2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu...","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \n...",500 m2,2 triệu/m2,Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Nhà đẹp 4x15m, Võ văn Vân, Vĩnh Lộc B, 525Tr/ căn",525 triệu- 60 m2,- 60 m2,"Đường Võ Văn Vân, Xã Vĩnh Lộc B, Huyện Bình Ch...","Nhà chính chủ mới xây đường võ văn vân, vĩnh l...",60 m²,"8,75 triệu/m²",Nam,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Đất TC 6x25m mặt tiền đường nhựa thôn 2 Suối Rao,440 triệu- 150 m2,- 150 m2,"Thôn 2, Xã Suối Rao, Huyện Châu Đức, Bà Rịa - ...","bán lô đất đẹp mặt tiền đường nhựa thôn 2, Suố...",150 m2,"2,93 triệu/m2",Bắc,Đã có sổ,Mặt tiền,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
